# 23. Referable Diabetic Retinopathy (RDR) Screening Calibration

This notebook demonstrates the clinical screening protocol for **Referable Diabetic Retinopathy (RDR)**:
- **Clinical Definition**: RDR corresponds to ICDRS Grade $\ge 2$ (Moderate NPDR, Severe NPDR, Proliferative DR).
- **Probability Aggregation**: The RDR risk score is derived from ordinal probabilities:
  $$p_{\text{RDR}} = \sum_{k=2}^{4} p_k = p_2 + p_3 + p_4$$
- **Zero-Leakage Calibration**: The operating decision threshold $\tau^*$ is calibrated to enforce a minimum clinical specificity target (typically $\ge 95\%$) **strictly on the source validation set** $\mathcal{D}_{\text{val}}^{\text{source}}$:
  $$\tau^* = \arg\max \{ \tau \in [0, 1] : \text{Specificity}(\tau; \mathcal{D}_{\text{val}}^{\text{source}}) \ge 0.95 \}$$
- **Generalization Evaluation**: The calibrated threshold $\tau^*$ is applied without alteration to held-out source test and out-of-domain target test cohorts.

In [ ]:
import numpy as np
from drdg.evaluation.rdr import calibrate_rdr_threshold, evaluate_rdr_screening

# Simulate source validation predictions (e.g. 500 patients)
np.random.seed(42)
n_val = 500
y_val_grades = np.random.choice([0, 1, 2, 3, 4], size=n_val, p=[0.5, 0.15, 0.2, 0.1, 0.05])

# Simulate model ordinal probabilities
logits = y_val_grades * 1.5 + np.random.randn(n_val)
p_rdr_val = 1.0 / (1.0 + np.exp(-logits))

# Calibrate operating threshold at >= 95% specificity on validation data
calib = calibrate_rdr_threshold(y_val_grades, p_rdr_val, min_specificity=0.95)
tau_star = calib["calibrated_threshold"]

print(f"Calibrated Threshold (tau*): {tau_star:.4f}")
print(f"Validation Operating Sensitivity: {calib['operating_sensitivity']:.4f}")
print(f"Validation Operating Specificity: {calib['operating_specificity']:.4f}")
print(f"Validation RDR AUC:               {calib['rdr_auc']:.4f}")

In [ ]:
# Evaluate calibrated threshold on unseen target test cohort (e.g., Messidor-2 or EyePACS)
n_test = 1000
y_test_grades = np.random.choice([0, 1, 2, 3, 4], size=n_test, p=[0.6, 0.1, 0.18, 0.08, 0.04])
logits_test = y_test_grades * 1.4 + np.random.randn(n_test) - 0.2  # domain shift
p_rdr_test = 1.0 / (1.0 + np.exp(-logits_test))

test_metrics = evaluate_rdr_screening(y_test_grades, p_rdr_test, threshold=tau_star)

print("Target Test Cohort Evaluation at Calibrated tau*:")
for k, v in test_metrics.items():
    print(f"  {k:24s}: {v:.4f}")

In [ ]:
# Plotting ROC Curve with Calibrated Operating Point
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

y_true_binary = (y_test_grades >= 2).astype(int)
fpr, tpr, _ = roc_curve(y_true_binary, p_rdr_test)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='#1f77b4', lw=2, label=f'Target Test ROC (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', label='Chance')

# Operating point
op_fpr = 1.0 - test_metrics['specificity']
op_tpr = test_metrics['sensitivity']
plt.scatter([op_fpr], [op_tpr], color='#d62728', s=100, zorder=5, 
            label=f'Calibrated Point (Sens={op_tpr:.2f}, Spec={test_metrics["specificity"]:.2f})')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=12)
plt.ylabel('True Positive Rate (Sensitivity)', fontsize=12)
plt.title('Referable DR Screening ROC & Calibrated Operating Point', fontsize=13)
plt.legend(loc='lower right', frameon=True)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()